In [9]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader
import os

In [2]:
loader = DirectoryLoader(
    "../data/rag_data",
    glob="**/*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"}
)

documents = loader.load()

print("Documents Loaded:", len(documents))

Documents Loaded: 10


In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

docs = splitter.split_documents(documents)

print("Chunks Created:", len(docs))

Chunks Created: 328


In [4]:
# =====================================================
# STEP 1: Import Embedding Model
# =====================================================

from langchain_huggingface import HuggingFaceEmbeddings

# =====================================================
# STEP 2: Load Embedding Model
# =====================================================

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding Model Loaded Successfully!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding Model Loaded Successfully!


In [5]:
sample_embedding = embeddings.embed_query(
    "What is SQL and Machine Learning Inner Join?"
)

print("Embedding Length:", len(sample_embedding))
print(sample_embedding[:10])

Embedding Length: 384
[-0.03745199739933014, -0.006339778658002615, 0.026095161214470863, 0.04259083420038223, 0.021674828603863716, -0.06687932461500168, 0.041051268577575684, -0.010063192807137966, -0.05297622084617615, -0.02090388908982277]


In [6]:
####Chroma DB
from langchain_community.vectorstores import Chroma

vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=embeddings,
    persist_directory="../vector_db"
)
print("Vector DB Created")
print(vectorstore._collection.count())

Vector DB Created
1640


In [7]:
###Testing RAG Retrieval
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 5,
        "fetch_k": 20
    }
)

results = retriever.invoke(
    "What is an INNER JOIN?"
)

for i, doc in enumerate(results, start=1):
    print(f"\nResult {i}")
    print("-"*50)
    print(doc.page_content[:500])


Result 1
--------------------------------------------------
```sql
SELECT e.emp_name, d.dept_name
FROM Employees e
FULL OUTER JOIN Departments d
ON e.dept_id = d.dept_id;
```

### Join Summary Table
| Join Type | Returned Rows |
|---|---|
| INNER JOIN | Only matching rows from both tables |
| LEFT JOIN | All rows from left table, matched rows from right |
| RIGHT JOIN | All rows from right table, matched rows from left |
| FULL OUTER JOIN | All rows from both tables, matched where possible |

### Interview Questions and Answers
**Q1. What is the differen

Result 2
--------------------------------------------------
## Quick Revision Notes
- A primary key uniquely identifies rows and cannot be null [1].
- A foreign key enforces valid references between tables [1].
- Constraints protect data integrity by rejecting invalid data [1].
- `INNER JOIN` returns matching rows, while outer joins preserve unmatched rows from one or both sides [3].
- `GROUP BY` is used with aggregate functions.
- `

In [10]:
from langchain_groq import ChatGroq
from dotenv import load_dotenv

load_dotenv()
groq_api_key = os.getenv("GROQ_API_KEY")

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=groq_api_key
)

In [11]:
###Testing LLM with RAG Retrieval
response = llm.invoke(
    "What is Python?"
)

print(response.content)

**Python Overview**

Python is a:

* **High-level programming language**: Easy to read and write, with a focus on code readability.
* **Interpreted language**: Code is executed line-by-line, without the need for compilation.
* **Multi-paradigm language**: Supports object-oriented, imperative, and functional programming styles.
* **Cross-platform language**: Can run on various operating systems, including Windows, macOS, and Linux.

**Key Features**
---------------

* **Easy to learn**: Simple syntax and concise code make it a great language for beginners.
* **Large standard library**: Includes modules for various tasks, such as file I/O, networking, and data analysis.
* **Extensive community**: Active and supportive community, with many online resources and libraries available.
* **Versatile**: Can be used for web development, scientific computing, data analysis, artificial intelligence, and more.

**Common Use Cases**
--------------------

* **Web development**: Frameworks like Django

In [12]:
question = "What is an INNER JOIN?"

results = retriever.invoke(question)

context = "\n\n".join(
    [doc.page_content for doc in results]
)

print(context[:1000])

```sql
SELECT e.emp_name, d.dept_name
FROM Employees e
FULL OUTER JOIN Departments d
ON e.dept_id = d.dept_id;
```

### Join Summary Table
| Join Type | Returned Rows |
|---|---|
| INNER JOIN | Only matching rows from both tables |
| LEFT JOIN | All rows from left table, matched rows from right |
| RIGHT JOIN | All rows from right table, matched rows from left |
| FULL OUTER JOIN | All rows from both tables, matched where possible |

### Interview Questions and Answers
**Q1. What is the difference between INNER JOIN and LEFT JOIN?**  
`INNER JOIN` returns only matching rows, while `LEFT JOIN` returns every row from the left table plus matches from the right [3].

**Q2. When would `FULL OUTER JOIN` be useful?**  
It is useful when both matched and unmatched records from both tables must be analyzed [3].

**Q3. Can joins be used on columns with different names?**  
Yes. Join conditions compare values, not necessarily identical column names [3].

## 6. GROUP BY

## Quick Revision Notes
- 

In [13]:
questions = [
    "What is an INNER JOIN?",
    "What topics are asked in TCS interviews?",
    "Explain polymorphism in Python"
]

for question in questions:

    results = retriever.invoke(question)

    context = "\n\n".join(
        [doc.page_content for doc in results]
    )

    prompt = f"""
    You are an AI Interview Preparation Assistant.

    Answer the user's question using ONLY the provided context.

    Context:
    {context}

    Question:
    {question}

    Answer:
    """

    response = llm.invoke(prompt)

    print("\n" + "="*80)
    print("QUESTION:", question)
    print("="*80)
    print(response.content)


QUESTION: What is an INNER JOIN?
`INNER JOIN` returns only matching rows from both tables.

QUESTION: What topics are asked in TCS interviews?
The topics asked in TCS interviews include operational questions such as "What do the first 90 days of success look like for someone stepping into this specific role?", technical questions like explaining TCP and UDP, bubble sort, and the difference between array and linked list, as well as behavioral questions that focus on communication, motivation, fit, and strengths and weaknesses.

QUESTION: Explain polymorphism in Python
Polymorphism in Python refers to the ability of an object to take on multiple forms. This can be achieved through two main types: compile-time polymorphism (method overloading) and runtime polymorphism (method overriding).

*   Compile-time polymorphism (method overloading) is not directly supported in Python. However, it can be emulated using default argument values or the `*args` syntax. In other languages, method overl

In [14]:
def ask_interview_assistant(question):

    results = retriever.invoke(question)

    context = "\n\n".join(
        [doc.page_content for doc in results]
    )

    prompt = f"""
    You are an AI Interview Preparation Assistant.

    Use the provided context to answer the question.

    If the context contains related information,
    explain the answer in a clear and professional way.

    Do not simply repeat the context.
    Expand and explain it.

    Context:
    {context}

    Question:
    {question}

    Answer:
    """

    response = llm.invoke(prompt)

    return response.content

In [15]:
print(
    ask_interview_assistant(
        "Explain polymorphism in Python?",
    )
)

Polymorphism is a fundamental concept in object-oriented programming (OOP) that allows for more flexibility and generic code. In Python, polymorphism refers to the ability of an object to take on multiple forms, depending on the context in which it is used. This can be achieved through method overriding or method overloading.

**Method Overriding (Runtime Polymorphism):**
In Python, method overriding occurs when a subclass provides a specific implementation for a method that is already defined in its parent superclass. The method resolution happens dynamically at runtime based on the actual object instance, not the reference variable type. This allows for more specific behavior in the subclass while still maintaining the same interface as the superclass.

**Method Overloading (Compile-time Polymorphism):**
Python also supports method overloading, also known as compile-time polymorphism, through a process called method signature resolution. This occurs when multiple methods within the s

In [16]:
print(
    ask_interview_assistant(
        "What topics are asked in Wipro interviews?",
    )
)

In Wipro interviews, a wide range of topics are covered to assess the candidate's technical, analytical, and communication skills. The technical interview typically focuses on core computer science concepts, programming skills, and knowledge of specific technologies. Some of the key topics that are commonly asked in Wipro technical interviews include:

1. **Programming fundamentals**: Data structures (arrays, linked lists, stacks, queues), algorithms (sorting, searching), and object-oriented programming (OOP) concepts.
2. **Computer Science fundamentals**: Operating Systems (OS), Computer Networks (CN), Database Management Systems (DBMS), and Software Engineering.
3. **Programming languages**: Proficiency in languages such as C++, Java, Python, and familiarity with their respective ecosystems.
4. **Data Structures and Algorithms**: Implementation and analysis of various data structures and algorithms, including time and space complexity.
5. **System design and architecture**: Designing